# L74 · ASR 错误分析 — 替换/删除/插入模式，从 WER 到可改进方向

**学习目标**
- 从对齐路径中提取 S/D/I 错误的具体词
- 统计最高频的错误模式（混淆矩阵）
- 识别哪类词对 ASR 最难（数字、专名、方言）

In [ ]:
import numpy as np
from collections import Counter

In [ ]:
# 复用 L67 的 alignment 函数
def alignment(ref_str, hyp_str):
    """返回词级别对齐操作列表：(op, ref_word, hyp_word)。op ∈ {M,S,I,D}"""
    a, b = ref_str.split(), hyp_str.split()
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1,m+1):
        for j in range(1,n+1):
            if a[i-1]==b[j-1]: dp[i][j]=dp[i-1][j-1]
            else: dp[i][j]=1+min(dp[i-1][j],dp[i][j-1],dp[i-1][j-1])
    ops,i,j=[],m,n
    while i>0 or j>0:
        if i>0 and j>0 and a[i-1]==b[j-1]:
            ops.append(('M',a[i-1],b[j-1])); i-=1; j-=1
        elif i>0 and j>0 and dp[i][j]==1+dp[i-1][j-1]:
            ops.append(('S',a[i-1],b[j-1])); i-=1; j-=1
        elif j>0 and dp[i][j]==1+dp[i][j-1]:
            ops.append(('I','-',b[j-1])); j-=1
        else:
            ops.append(('D',a[i-1],'-')); i-=1
    return list(reversed(ops))

def wer(ref, hyp):
    a, b = ref.lower().split(), hyp.lower().split()
    if not a: return 0.0
    m,n=len(a),len(b)
    dp=[[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0]=i
    for j in range(n+1): dp[0][j]=j
    for i in range(1,m+1):
        for j in range(1,n+1):
            dp[i][j]=dp[i-1][j-1] if a[i-1]==b[j-1] else 1+min(dp[i-1][j],dp[i][j-1],dp[i-1][j-1])
    return dp[m][n]/len(a)

## 10 对 (ref, hyp) 样本

In [ ]:
PAIRS = [
    ("the quick brown fox jumps over the lazy dog",
     "the quick brown fox jump over the lazy dog"),
    ("she sells seashells by the seashore",
     "she cells seashells by the seashore"),
    ("call nine one one immediately",
     "call 911 immediately"),
    ("the meeting is scheduled for three pm on friday",
     "the meeting is scheduled for 3 pm on friday"),
    ("artificial intelligence is transforming healthcare",
     "artificial intelligence is transforming health care"),
    ("please send the report to john smith",
     "please send the report to john's myth"),
    ("the temperature is twenty three degrees celsius",
     "the temperature is 23 degrees celsius"),
    ("i would like a cup of coffee please",
     "i would like a cup of coffee"),
    ("deep learning models require large amounts of training data",
     "deep learning models require large amounts of training data"),
    ("natural language processing has many applications",
     "natural language process has many application"),
]

print(f'{"#":>2}  {"WER":>5}  REF')
print('-' * 55)
for i, (ref, hyp) in enumerate(PAIRS):
    w = wer(ref, hyp)
    print(f'{i:>2}  {w:>5.1%}  {ref[:45]}')

## 错误统计

In [ ]:
sub_counter = Counter()
del_counter = Counter()
ins_counter = Counter()

for ref, hyp in PAIRS:
    for op, rw, hw in alignment(ref, hyp):
        if op == 'S':
            sub_counter[(rw, hw)] += 1
        elif op == 'D':
            del_counter[rw] += 1
        elif op == 'I':
            ins_counter[hw] += 1

print('=== 最常见替换 (ref→hyp) ===')
for (r, h), cnt in sub_counter.most_common(8):
    print(f'  {r:15s} → {h:15s}  ×{cnt}')

print()
print('=== 最常见删除 ===')
for word, cnt in del_counter.most_common(5):
    print(f'  deleted: {word:15s}  ×{cnt}')

print()
print('=== 最常见插入 ===')
for word, cnt in ins_counter.most_common(5):
    print(f'  inserted: {word:15s}  ×{cnt}')

## 错误模式总结

| 错误类型 | 例子 | 根本原因 | 改进方向 |
|---|---|---|---|
| 数字/文字混淆 | nine one one → 911 | 训练数据格式不统一 | 后处理规范化 |
| 形近词替换 | sells → cells | 声学特征相似 | 更多上下文 / LM |
| 专有名词 | john smith → john's myth | OOV 词汇 | 热词增强 / 微调 |
| 尾词删除 | coffee please → coffee | 句末能量低 | VAD 优化 |

**下一步 L75**：把这些分析做成可视化仪表板。